In [13]:
using Pkg;
Pkg.activate(".");
Pkg.add("Plots");
Pkg.add("BenchmarkTools");
using Plots
using BenchmarkTools

  Activating project at `~/nbodyjulia`
   Resolving package versions...
     Project No packages added to or removed from `~/nbodyjulia/Project.toml`
    Manifest No packages added to or removed from `~/nbodyjulia/Manifest.toml`
   Resolving package versions...
    Updating `~/nbodyjulia/Project.toml`
  [6e4b80f9] + BenchmarkTools v1.8.0
    Updating `~/nbodyjulia/Manifest.toml`
  [6e4b80f9] + BenchmarkTools v1.8.0
  [34da2185] + Compat v4.18.1
  [9abbd945] + Profile v1.11.0
[ Info: Precompiling BenchmarkTools [6e4b80f9-dd63-53aa-95a3-0cdb28fa8baf](cache misses: wrong dep version loaded (3))
[ Info: Precompiling BenchmarkTools [6e4b80f9-dd63-53aa-95a3-0cdb28fa8baf] (cache misses: wrong dep version loaded (6))

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up


# N-Body simulation
The following simple example code simulates the trajectory of celestial bodies that accelerate each other via gravity.

You can learn more about it here: [https://en.wikipedia.org/wiki/N-body_simulation](https://en.wikipedia.org/wiki/N-body_simulation)

In [2]:
solarsize = [13910, 4880, 12104, 12742, 6779, 139820, 116460, 50724, 49244];

dt = 86400;
G  = 6.67e-11;

# Each point has a position, a velocity and a mass.
# The mass (ideally) never changes.
mutable struct Point
    pos
    vel
    mass
end

# Print a point for debugging purposes
function printP(p)
    for v in p
        println(v);
    end
end

# Generate a number of random objects
function gen(type, num, max_x=100.0, max_y=100.0, max_mass=1000.0)
    v = Vector(undef, num)
    for ix in 1:num
        v[ix] = Point((rand()*max_x, rand()*max_y), (0.0,0.0), rand()*max_mass)
    end
    return v
end

# Creates a simple solar system example
function solar(T)
    v = Vector{Point}(undef, 9);
    v[1] = Point(( 0.0,0.0),        (0.0,0.0,),      1.989e30 );   # sun
    v[2] = Point(( 57.909e9,0.0),   (0.0,47.36e3,),  0.33011e24 ); # mercury
    v[3] = Point(( 108.209e9,0.0),  (0.0,35.02e3,),  4.8675e24 );  # venus
    v[4] = Point(( 149.596e9,0.0),  (0.0,29.78e3,),  5.9724e24 );  # earth
    v[5] = Point(( 227.923e9,0.0),  (0.0,24.07e3,),  0.64171e24 ); # mars
    v[6] = Point(( 778.570e9,0.0),  (0.0,13e3,),     1898.19e24 ); # jupiter
    v[7] = Point(( 1433.529e9,0.0), (0.0,9.68e3,),   568.34e24 );  # saturn
    v[8] = Point(( 2872.463e9,0.0), (0.0,6.80e3,),   86.813e24 );  # uranus
    v[9] = Point(( 4495.060e9,0.0), (0.0,5.43e3,),   102.413e24 ); # neptune
    return v;
end

# Simulate one time step
function simulation_step(v)
    # array for temporary forces
    forces = zeros(size(v,1), 2);
    # compute force influence on every object,
    # by accumulating the influence by every other object
    for iq in eachindex(v) # for every object
        q = v[iq];
        total_a = (0.0, 0.0); # force accumulation
        for ik in eachindex(v) # for every other object
            if iq == ik # except the object itself
                continue
            end
            k = v[ik];
            distance = k.pos .- q.pos; # distance between objects
            mag = sqrt(distance[1] * distance[1] + distance[2] * distance[2]);
            acceleration = 1.0 * G * q.mass * k.mass / mag^3; # acceleration
            total_a = total_a .+ (distance .* acceleration);
        end
        forces[iq,:] .= total_a; # save total force to temporary array
    end
    # update all positions and velocities
    for iq in eachindex(v)
        q = v[iq];
        q.pos = q.pos .+ (q.vel  .* dt);
        force = (forces[iq,1], forces[iq,2]);
        q.vel = q.vel .+ ((force ./ q.mass) .* dt);
    end
end


simulation_step (generic function with 1 method)

In [10]:
function plotv(v)
    minx = -5000e9;
    maxx = 5000e9;
    miny = -5000e9;
    maxy = 5000e9;
    p = scatter(map(x->x.pos[1], v),map(x->x.pos[2], v), label="", xlim=(minx, maxx), ylim=(miny, maxy), size=(500,500), markersize=solarsize./5000)
end

plotv (generic function with 1 method)

In [11]:
function plot_simulation()
    v = solar(Float64);
    plotv(v);
    anim = @animate for i in 1:200
        for j in 1:500
            simulation_step(v);
        end
        plotv(v);
    end
    return mp4(anim, "nbody.mp4", fps=5)
end
plot_simulation()

main (generic function with 1 method)

In [15]:
function benchmark_simulation()
    v = gen(Float64, 100);
    @benchmark for i in 1:100
        simulation_step($v)
    end
end
benchmark_simulation()

BenchmarkTools.Trial: 6 samples with 1 evaluation per sample.
 Range (min … max):  858.991 ms … 867.545 ms  ┊ GC (min … max): 10.38% … 10.30%
 Time  (median):     862.946 ms               ┊ GC (median):    10.34%
 Time  (mean ± σ):   862.932 ms ±   3.216 ms  ┊ GC (mean ± σ):  10.33% ±  0.04%

  ▁         ▁     ▁                       █                   ▁  
  █▁▁▁▁▁▁▁▁▁█▁▁▁▁▁█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁█ ▁
  859 ms           Histogram: frequency by time          868 ms <

 Memory estimate: 352.64 MiB, allocs estimate: 15980200.

# Performance engineering ideas
Where to go from here?

# Serial code optimization

Look at the code and think about some of the optimizations we discussed.
Create a new version of the code, maybe in a copy of the notebook, do an optimization and benchmark again.
Did it become faster? Why not?
You can try some profiling or other analysis technique to get an idea what to improve.

You can also vary the benchmark:
Measure the execution time depending on the
* number of objects
* number of simulation steps

Record the measurements (without influencing the measurement) and plot them.
Maybe you find curious behavior.

# Parallelization

Can you think about a way to parallelize the simulation?
Which part of the work is done by each thread?
Which results need to be exchanged?

This code serves as a simple example to play around with and apply some of the techniques we presented.
Have fun!